# 04 · Redes neuronales

## Objetivo

En este notebook vamos a introducir las redes neuronales a partir de una pregunta muy concreta:

> ¿Qué ocurre cuando la frontera que separa dos clases no es lineal?

Hasta ahora hemos trabajado con modelos relativamente simples:

- regresión lineal;
- regresión logística;
- árboles de decisión;
- validación y overfitting.

Ahora veremos una **red neuronal multicapa** usando `MLPClassifier` de `scikit-learn`.

No vamos a entrar todavía en PyTorch o TensorFlow. El objetivo aquí es entender bien las ideas fundamentales:

- neuronas;
- pesos y sesgos;
- funciones de activación;
- capas ocultas;
- propagación hacia delante;
- función de pérdida;
- ajuste iterativo de pesos;
- necesidad de escalar variables;
- capacidad del modelo;
- underfitting y overfitting en redes.

Este notebook será el puente conceptual hacia Deep Learning.


## 1. Importaciones


In [ ]:
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
)

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)


# 2. Crear un problema no lineal

Usaremos un dataset sintético con dos clases en forma de "lunas".

Podemos imaginarlo como un problema ATM simplificado con dos variables abstractas:

- `traffic_complexity`;
- `coordination_pressure`.

El target indica:

- `0`: situación operativa tipo A;
- `1`: situación operativa tipo B.

Lo importante no es el significado exacto de las variables, sino que la separación entre clases **no puede describirse bien con una línea recta**.


In [ ]:
X, y = make_moons(
    n_samples=700,
    noise=0.22,
    random_state=42
)

df = pd.DataFrame(
    X,
    columns=["traffic_complexity", "coordination_pressure"]
)

df["class"] = y

df.head()


In [ ]:
plt.figure(figsize=(8, 6))

for cls in [0, 1]:
    subset = df[df["class"] == cls]
    plt.scatter(
        subset["traffic_complexity"],
        subset["coordination_pressure"],
        alpha=0.65,
        label=f"Clase {cls}"
    )

plt.xlabel("traffic_complexity")
plt.ylabel("coordination_pressure")
plt.title("Problema de clasificación no lineal")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


A simple vista se aprecia que una sola línea recta no puede separar perfectamente ambas clases.

Eso es importante porque una regresión logística básica aprende una frontera de decisión lineal.


## 3. Train / test split


In [ ]:
X = df[["traffic_complexity", "coordination_pressure"]]
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)


# 4. Baseline: regresión logística

Antes de usar una red neuronal necesitamos un baseline.

La regresión logística aprenderá una frontera:

$$
b + w_1x_1 + w_2x_2 = 0
$$

Es decir, una recta en este espacio bidimensional.


In [ ]:
logistic_model = LogisticRegression()

logistic_model.fit(X_train, y_train)

logistic_pred = logistic_model.predict(X_test)

print(
    "Accuracy regresión logística:",
    round(accuracy_score(y_test, logistic_pred), 3)
)


In [ ]:
print(classification_report(y_test, logistic_pred))


## 5. Función auxiliar para visualizar fronteras de decisión

Esta función nos permitirá ver qué regiones del espacio asigna un modelo a cada clase.


In [ ]:
def plot_decision_boundary(model, X, y, title, step=0.03):
    x_min, x_max = X.iloc[:, 0].min() - 0.6, X.iloc[:, 0].max() + 0.6
    y_min, y_max = X.iloc[:, 1].min() - 0.6, X.iloc[:, 1].max() + 0.6

    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, step),
        np.arange(y_min, y_max, step)
    )

    grid = pd.DataFrame({
        X.columns[0]: xx.ravel(),
        X.columns[1]: yy.ravel()
    })

    zz = model.predict(grid).reshape(xx.shape)

    plt.figure(figsize=(8, 6))
    plt.contourf(xx, yy, zz, alpha=0.25)

    for cls in np.unique(y):
        mask = y == cls
        plt.scatter(
            X.loc[mask, X.columns[0]],
            X.loc[mask, X.columns[1]],
            alpha=0.65,
            label=f"Clase {cls}"
        )

    plt.xlabel(X.columns[0])
    plt.ylabel(X.columns[1])
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()


In [ ]:
plot_decision_boundary(
    logistic_model,
    X_train,
    y_train,
    "Frontera de decisión: regresión logística"
)


Observa que la frontera es esencialmente lineal.

El modelo intenta hacerlo lo mejor posible, pero la geometría del problema requiere algo más flexible.


# 6. Primera red neuronal

Usaremos un **Multi-Layer Perceptron (MLP)**.

Arquitectura inicial:

```text
2 entradas
   ↓
8 neuronas ocultas
   ↓
1 salida de clasificación
```

En `MLPClassifier` indicamos la capa oculta mediante:

```python
hidden_layer_sizes=(8,)
```


In [ ]:
mlp = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(
        hidden_layer_sizes=(8,),
        activation="relu",
        solver="adam",
        max_iter=2000,
        random_state=42
    ))
])

mlp.fit(X_train, y_train)

mlp_pred = mlp.predict(X_test)

print(
    "Accuracy red neuronal:",
    round(accuracy_score(y_test, mlp_pred), 3)
)


In [ ]:
print(classification_report(y_test, mlp_pred))


In [ ]:
plot_decision_boundary(
    mlp,
    X_train,
    y_train,
    "Frontera de decisión: red neuronal"
)


La diferencia importante es que la red neuronal puede construir una frontera **no lineal**.

No lo hace porque conozca explícitamente la forma de las lunas.

Lo hace porque combina varias transformaciones aprendidas.


# 7. ¿Qué es una neurona?

Una neurona artificial realiza esencialmente dos pasos.

Primero calcula una combinación lineal:

$$
z = w_1x_1 + w_2x_2 + \dots + w_nx_n + b
$$

Después aplica una función de activación:

$$
a = f(z)
$$

Los elementos son:

- $x_i$: entradas;
- $w_i$: pesos;
- $b$: bias o sesgo;
- $f$: función de activación;
- $a$: salida de la neurona.

Sin función de activación, encadenar muchas capas lineales seguiría produciendo una transformación lineal.


# 8. La función ReLU

Una activación habitual es ReLU:

$$
ReLU(z)=\max(0,z)
$$

Es extremadamente simple:

- valores negativos → 0;
- valores positivos → se mantienen.


In [ ]:
z = np.linspace(-5, 5, 300)
relu = np.maximum(0, z)

plt.figure(figsize=(8, 5))
plt.plot(z, relu)
plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)
plt.xlabel("z")
plt.ylabel("ReLU(z)")
plt.title("Función de activación ReLU")
plt.grid(alpha=0.2)
plt.show()


Lo importante no es que ReLU sea una función especialmente sofisticada.

Lo importante es que introduce **no linealidad**.

Gracias a eso, varias neuronas y varias capas pueden representar funciones mucho más complejas que una única transformación lineal.


# 9. ¿Por qué necesitamos escalar los datos?

Las redes neuronales suelen entrenarse mejor cuando las variables tienen escalas comparables.

Por eso hemos usado:

```python
StandardScaler()
```

La transformación aproximada es:

$$
x_{scaled} = \frac{x-\mu}{\sigma}
$$

Esto produce variables con media próxima a 0 y desviación estándar próxima a 1.


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns
)

scaled_df.describe().round(3)


Importante:

```text
fit del scaler → solo sobre train
transform      → train y test
```

El `Pipeline` de scikit-learn hace esto correctamente y ayuda a evitar data leakage.


# 10. Qué ha aprendido la red

Podemos inspeccionar el modelo interno.


In [ ]:
trained_mlp = mlp.named_steps["mlp"]

print("Capas:")
print(trained_mlp.n_layers_)

print()
print("Número de iteraciones:")
print(trained_mlp.n_iter_)

print()
print("Forma de las matrices de pesos:")
for i, weights in enumerate(trained_mlp.coefs_):
    print(f"Capa {i}: {weights.shape}")

print()
print("Forma de los vectores de bias:")
for i, bias in enumerate(trained_mlp.intercepts_):
    print(f"Capa {i}: {bias.shape}")


Con dos variables de entrada y ocho neuronas ocultas esperamos aproximadamente:

```text
entrada (2)
   ↓
pesos 2 x 8
   ↓
capa oculta (8)
   ↓
pesos 8 x 1
   ↓
salida
```

Cada conexión tiene un peso aprendido.


# 11. Función de pérdida

Durante el entrenamiento la red modifica los pesos para reducir una función de pérdida.

En clasificación binaria se utiliza una variante de **log loss / cross-entropy**.

Una forma simplificada es:

$$
L = -[y\log(p)+(1-y)\log(1-p)]
$$

Si el valor real es `1` pero el modelo asigna una probabilidad muy pequeña a `1`, la pérdida será grande.

Si asigna una probabilidad alta a la clase correcta, la pérdida será pequeña.


## 12. Evolución de la pérdida durante el entrenamiento

`MLPClassifier` guarda la pérdida observada durante las iteraciones.


In [ ]:
loss_curve = trained_mlp.loss_curve_

plt.figure(figsize=(8, 5))
plt.plot(loss_curve)
plt.xlabel("Iteración")
plt.ylabel("Loss")
plt.title("Evolución de la función de pérdida")
plt.grid(alpha=0.2)
plt.show()

print("Loss inicial:", round(loss_curve[0], 4))
print("Loss final  :", round(loss_curve[-1], 4))


El patrón general que buscamos es:

```text
pesos iniciales
      ↓
predicción
      ↓
cálculo de loss
      ↓
cálculo de gradientes
      ↓
actualización de pesos
      ↓
nueva predicción
      ↓
...
```

En una red neuronal este proceso se realiza repetidamente.

La técnica para propagar el error hacia atrás y calcular gradientes se llama **backpropagation**.


# 13. Comparar distintas capacidades

Vamos a probar varias arquitecturas:

```text
(1,)       → una neurona oculta
(2,)       → dos neuronas ocultas
(4,)
(8,)
(16,)
(32,)
(32, 16)   → dos capas ocultas
```

La red tendrá más capacidad a medida que aumentemos neuronas y capas.


In [ ]:
architectures = [
    (1,),
    (2,),
    (4,),
    (8,),
    (16,),
    (32,),
    (32, 16),
]

rows = []

for architecture in architectures:
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPClassifier(
            hidden_layer_sizes=architecture,
            activation="relu",
            solver="adam",
            max_iter=2500,
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    rows.append({
        "architecture": str(architecture),
        "train_accuracy": accuracy_score(y_train, train_pred),
        "test_accuracy": accuracy_score(y_test, test_pred),
    })

architecture_results = pd.DataFrame(rows)
architecture_results


In [ ]:
x = np.arange(len(architecture_results))

plt.figure(figsize=(10, 5))
plt.plot(
    x,
    architecture_results["train_accuracy"],
    marker="o",
    label="Train"
)
plt.plot(
    x,
    architecture_results["test_accuracy"],
    marker="o",
    label="Test"
)

plt.xticks(
    x,
    architecture_results["architecture"],
    rotation=45
)
plt.xlabel("Arquitectura")
plt.ylabel("Accuracy")
plt.title("Capacidad de la red")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


Más capacidad no significa automáticamente mejor generalización.

Una red muy pequeña puede sufrir **underfitting**.

Una red excesivamente flexible puede acabar capturando ruido y sufrir **overfitting**.

El concepto es exactamente el mismo que vimos con árboles de decisión.


# 14. Una red deliberadamente grande

Entrenaremos una red mucho mayor de la necesaria para este problema.


In [ ]:
large_mlp = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(
        hidden_layer_sizes=(128, 128, 64),
        activation="relu",
        solver="adam",
        alpha=0.00001,
        max_iter=3000,
        random_state=42
    ))
])

large_mlp.fit(X_train, y_train)

large_train_pred = large_mlp.predict(X_train)
large_test_pred = large_mlp.predict(X_test)

print(
    "Train accuracy:",
    round(accuracy_score(y_train, large_train_pred), 3)
)
print(
    "Test accuracy :",
    round(accuracy_score(y_test, large_test_pred), 3)
)


In [ ]:
plot_decision_boundary(
    large_mlp,
    X_train,
    y_train,
    "Frontera de decisión: red neuronal grande"
)


Observa la frontera.

Una frontera más complicada no es necesariamente mejor.

La pregunta relevante sigue siendo:

> ¿generaliza a datos nuevos?


# 15. Regularización

`MLPClassifier` incluye regularización L2 mediante el parámetro:

```python
alpha
```

Un `alpha` mayor penaliza pesos grandes y puede reducir overfitting.


In [ ]:
alphas = [0.00001, 0.001, 0.01, 0.1, 1.0]

rows = []

for alpha in alphas:
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPClassifier(
            hidden_layer_sizes=(32, 16),
            activation="relu",
            solver="adam",
            alpha=alpha,
            max_iter=2500,
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)

    rows.append({
        "alpha": alpha,
        "train_accuracy": accuracy_score(
            y_train,
            model.predict(X_train)
        ),
        "test_accuracy": accuracy_score(
            y_test,
            model.predict(X_test)
        ),
    })

regularization_results = pd.DataFrame(rows)
regularization_results


La regularización intenta reducir la tendencia del modelo a utilizar pesos excesivamente grandes para adaptarse demasiado a los datos de entrenamiento.


# 16. Early stopping

Otra técnica habitual es detener el entrenamiento cuando el rendimiento deja de mejorar.

`MLPClassifier` permite:

```python
early_stopping=True
```

Internamente reserva una parte del conjunto de entrenamiento como validación.


In [ ]:
early_stop_mlp = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(
        hidden_layer_sizes=(32, 16),
        activation="relu",
        solver="adam",
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=20,
        max_iter=3000,
        random_state=42
    ))
])

early_stop_mlp.fit(X_train, y_train)

early_model = early_stop_mlp.named_steps["mlp"]

print("Iteraciones realizadas:", early_model.n_iter_)
print(
    "Test accuracy:",
    round(
        accuracy_score(
            y_test,
            early_stop_mlp.predict(X_test)
        ),
        3
    )
)


Early stopping es una idea que volverá a aparecer con frecuencia en Deep Learning:

```text
entrenar
   ↓
comprobar validation
   ↓
seguir mientras mejore
   ↓
parar cuando deja de mejorar
```


# 17. ¿Qué hace realmente una capa oculta?

Una forma útil de pensarlo es:

```text
variables originales
        ↓
combinaciones aprendidas
        ↓
activaciones no lineales
        ↓
nuevas representaciones
        ↓
clasificación
```

La red aprende internamente una nueva representación de los datos.

En problemas grandes, esas representaciones pueden ser extremadamente complejas.

Esta capacidad de aprender **representaciones** es una de las razones fundamentales por las que el Deep Learning es tan potente.


# 18. De una red pequeña a Deep Learning

Una red con una sola capa oculta:

```text
input
  ↓
hidden
  ↓
output
```

es una red neuronal.

Cuando acumulamos muchas capas:

```text
input
  ↓
hidden 1
  ↓
hidden 2
  ↓
hidden 3
  ↓
...
  ↓
output
```

hablamos habitualmente de **Deep Neural Networks**.

Por eso:

```text
Machine Learning
└── Neural Networks
    └── Deep Learning
```

Deep Learning no es algo separado de Machine Learning: es una familia de técnicas dentro de él.


# 19. Por qué no usamos todavía PyTorch

Para entender redes neuronales, `MLPClassifier` tiene una ventaja:

- permite concentrarnos en conceptos;
- evita código de bajo nivel;
- mantiene una interfaz similar a otros modelos de scikit-learn.

Más adelante, con PyTorch, tendremos que manejar explícitamente:

- tensores;
- capas;
- forward pass;
- loss;
- gradientes;
- optimizer;
- batches;
- epochs;
- GPU.

Eso tendrá mucho más sentido una vez que estas ideas básicas estén claras.


# 20. Ejercicios

### Ejercicio 1

Cambia:

```python
noise=0.22
```

por:

```python
noise=0.05
```

¿Qué ocurre con la accuracy?

### Ejercicio 2

Prueba:

```python
hidden_layer_sizes=(2,)
```

y después:

```python
hidden_layer_sizes=(64, 64)
```

Compara las fronteras.

### Ejercicio 3

Elimina `StandardScaler`.

¿Cambia el entrenamiento?

Prueba también a multiplicar una de las variables por 1000 para que las escalas sean muy diferentes.

### Ejercicio 4

Prueba las activaciones:

```python
activation="tanh"
activation="logistic"
activation="relu"
```

¿Obtienes la misma frontera?

### Ejercicio 5

Reduce el entrenamiento a 100 muestras.

¿Es más fácil que una red grande sobreajuste?

### Ejercicio 6

Explica por qué una red neuronal con varias capas pero sin funciones de activación no lineales seguiría siendo equivalente a una transformación lineal.


# 21. Ideas clave

Al terminar este notebook deberías poder explicar:

1. Por qué una regresión logística tiene una frontera lineal.
2. Qué problema resuelve una red neuronal en este ejemplo.
3. Qué hace una neurona artificial.
4. Qué son pesos y bias.
5. Para qué sirve una función de activación.
6. Por qué ReLU introduce no linealidad.
7. Qué significa una capa oculta.
8. Por qué conviene escalar las variables.
9. Qué representa la función de pérdida.
10. Qué significa backpropagation a nivel conceptual.
11. Por qué más neuronas o más capas aumentan la capacidad.
12. Cómo una red puede sufrir underfitting u overfitting.
13. Qué hacen la regularización y early stopping.
14. Qué relación existe entre redes neuronales y Deep Learning.

## La idea central

Una red neuronal aprende una sucesión de transformaciones:

```text
datos
  ↓
combinaciones lineales
  ↓
activaciones no lineales
  ↓
representaciones internas
  ↓
predicción
```

## Siguiente paso

Con esto cerramos el bloque fundamental de ML que queríamos consolidar mediante notebooks.

El siguiente salto ya puede ser hacia **Deep Learning de forma más explícita**, donde veremos cómo se construye y entrena una red con un framework como PyTorch y qué cambian conceptos como tensores, batches, epochs, optimizers y GPU.
